<a href="https://colab.research.google.com/github/mironovromandenisovich-creator/tetstst/blob/main/notebooks/comfyui_colab_with_manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [1]:
# ==========================================
# 📦 0. УСТАНОВКА БАЗОВЫХ ПАКЕТОВ (ДО ИМПОРТОВ)
# ==========================================
!pip install -q GitPython sageattention
!apt-get install -y -qq aria2 > /dev/null 2>&1

# Скачиваем и устанавливаем официальный бинарник Cloudflared
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import os
import re
import time
import socket
import threading
import subprocess

# ==========================================
# ⚙️ 1. ТОКЕНЫ И НАСТРОЙКИ
# ==========================================
CIVITAI_TOKEN = "27950b59d0ab3d466f24b4f798c7197c"
PORT = 8188

def add_token(url, token):
    sep = "&" if "?" in url else "?"
    return f"{url}{sep}token={token}"

# ==========================================
# 🚀 2. ПОДГОТОВКА СИСТЕМЫ И ПЛАГИНОВ
# ==========================================
print("🚀 [1/5] Проверка ядра ComfyUI и кастомных нод...")

# ComfyUI Core
if not os.path.exists("/content/ComfyUI/main.py"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
else:
    %cd /content/ComfyUI
    !git pull

# ComfyUI-Manager
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
else:
    %cd /content/ComfyUI/custom_nodes/ComfyUI-Manager
    !git pull

# Установка зависимостей ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# ==========================================
# 📁 3. СОЗДАНИЕ СТРУКТУРЫ ПАПОК
# ==========================================
print("\n📁 [2/5] Создание рабочих директорий...")
os.makedirs("/content/ComfyUI/models/loras", exist_ok=True)
os.makedirs("/content/ComfyUI/models/diffusion_models", exist_ok=True)
os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
os.makedirs("/content/ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("/content/ComfyUI/models/vae", exist_ok=True)
os.makedirs("/content/ComfyUI/models/upscale_models", exist_ok=True)

# ==========================================
# 📥 4. СКАЧИВАНИЕ МОДЕЛЕЙ И LORA
# ==========================================
print("\n📥 [3/5] Загрузка моделей и LoRA...")

# --- Б. Базовые энкодеры для оставшихся моделей картинок (HuggingFace) ---
print("  -> Проверяем Qwen3-VL и VAE...")
!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/text_encoders/qwen3vl_4b_bf16.safetensors" \
  -d /content/ComfyUI/models/text_encoders/ -o qwen3vl_4b_bf16.safetensors

!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/vae/qwen_image_vae.safetensors" \
  -d /content/ComfyUI/models/vae/ -o qwen_image_vae.safetensors

# --- В. Апскейлер RealESRGAN ---
print("  -> Проверяем апскейлер RealESRGAN...")
!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/spaces/Marne/Real-ESRGAN/resolve/main/RealESRGAN_x4plus.pth" \
  -d /content/ComfyUI/models/upscale_models/ -o RealESRGAN_x4plus.pth

# --- Г. Все 4 LoRA с Civitai ---
print("  -> Скачиваем LoRA с Civitai...")
loras = [
    "https://civitai.com/api/download/models/3090634?fileId=2970148",  # Krea2-realism-V2
    "https://civitai.com/api/download/models/3071904?fileId=2950820",  # HMNSFW
    "https://civitai.com/api/download/models/3290120?fileId=3174557",  # KNP LoRA
    "https://civitai.com/api/download/models/3147117?fileId=3027612"   # SNOFS Uncensored
]
for url in loras:
    auth_url = add_token(url, CIVITAI_TOKEN)
    !aria2c --content-disposition=true --console-log-level=warn -c -x 16 -s 16 -k 1M -d /content/ComfyUI/models/loras "{auth_url}"

# --- Д. Оставшиеся отборные модели картинок с Civitai ---
print("  -> Скачиваем модели картинок (Redcraft, Dark Beast, Anime v2, 3064584)...")
models = [
    "https://civitai.com/api/download/models/3139241?fileId=3259640",  # Redcraft Hybrid
    "https://civitai.com/api/download/models/3365025?fileId=3255345",  # Dark Beast 24GB
    "https://civitai.com/api/download/models/3357716?fileId=3246945",  # Krea 2 Anime v2.0
    "https://civitai.com/api/download/models/3064584?fileId=2943406"   # Модель 3064584
]
for url in models:
    auth_url = add_token(url, CIVITAI_TOKEN)
    !aria2c --content-disposition=true --console-log-level=warn -c -x 16 -s 16 -k 1M -d /content/ComfyUI/models/diffusion_models "{auth_url}"

# Симлинки в папку checkpoints (чтобы все модели были видны везде)
!ln -sf /content/ComfyUI/models/diffusion_models/* /content/ComfyUI/models/checkpoints/ 2>/dev/null || true

# ==========================================
# 🌐 5. ЗАПУСК CLOUDFLARE И COMFYUI
# ==========================================
print("\n🌐 [4/5] Подготовка туннеля Cloudflare...")
!pkill -f "main.py"
!pkill -f "cloudflared"

def wait_for_port_and_launch(port):
    # Ждём запуска локального сервера
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break

    print("\n" + "=" * 65)
    print("✅ СЕРВЕР COMFYUI УСПЕШНО ЗАПУЩЕН! ПОДКЛЮЧАЕМ CLOUDFLARE...")

    log_path = "/content/cloudflared.log"
    if os.path.exists(log_path):
        os.remove(log_path)

    # Запускаем туннель Cloudflare в фоне и пишем лог в файл
    cmd = f"cloudflared tunnel --url http://127.0.0.1:{port} > {log_path} 2>&1"
    subprocess.Popen(cmd, shell=True)

    # Ищем сгенерированную ссылку в логе
    url = None
    for _ in range(40):
        time.sleep(1)
        if os.path.exists(log_path):
            with open(log_path, "r") as f:
                content = f.read()
                match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
                if match:
                    url = match.group(0)
                    break

    if url:
        print(f"👉 ВАША ССЫЛКА НА ИНТЕРФЕЙС: {url}")
    else:
        print("⚠️ Не удалось получить ссылку автоматически. Проверьте лог командой: !cat /content/cloudflared.log")
    print("=" * 65 + "\n")
    print("💡 Совет: для параллельной загрузки используйте кнопку >_ Terminal в левом нижнем углу!")

# Запускаем фоновый мониторинг порта и туннеля
threading.Thread(target=wait_for_port_and_launch, daemon=True, args=(PORT,)).start()

print("\n🚀 [5/5] Запуск ComfyUI...")
%cd /content/ComfyUI
!python main.py --dont-print-server

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.9 MB/s eta 0:00:00
🚀 [1/5] Проверка ядра ComfyUI и кастомных нод...
Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 55062, done.
remote: Counting objects: 100% (611/611), done.
remote: Compressing objects: 100% (218/218), done.
remote: Total 55062 (delta 551), reused 394 (delta 393), pack-reused 54451 (from 3)
Receiving objects: 100% (55062/55062), 92.94 MiB | 19.47 MiB/s, done.
Resolving deltas: 100% (37534/37534), done.
Cloning into '/content/ComfyUI/custom_nodes/ComfyUI-Manager'...
remote: Enumerating objects: 32926, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 32926 (delta 63), reused 59 (delta 46), pack-reused 32838 (from 2)
Receiving objects: 100% (32926/32926), 191.59 MiB | 14.19 MiB/s, done.
Resolving deltas: 100% (24388/24388), done.
/content/ComfyUI
   ━

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname


[START] Security scan


[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-03 17:06:43.360
** Platform: Linux
** Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log


[INFO] 
Prestartup times for custom nodes:
[INFO]    5.6 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager
[INFO] 


...

[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-03 17:47:08.309
** Platform: Linux
** Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log


[INFO] 
Prestartup times for custom nodes:
[INFO]    0.0 seconds: /content/ComfyUI/custom_nodes/rgthree-comfy
[INFO]    0.0 seconds: /content/ComfyUI/custom_nodes/comfyui-easy-use
[INFO]    5.7 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager
[INFO] 


[INFO] Found comfy_kitchen backend hip: {'available': False, 'disabled': False, 'unavailable_reason': 'PyTorch ROCm/HIP runtime not available', 'capabilities': []}
[INFO] Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'int8_linear', 'na3d', 'quantize_and_rotate_rowwise', 'quantize_int8_rowwise', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'rms_adaln', 'rms_rope', 'rms_rope1', 'rms_rope1_', 'rms_rope_', 'rms_rope_split_half', 'rms_rope_split_half1', 'rms_rope_split_half1_', 'rms_rope_split_half_', 'w4a8_int8_linear']}
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', '

[INFO] aimdo: /project/src/control.c:283:INFO:comfy-aimdo inited for GPU: NVIDIA A100-SXM4-80GB (VRAM: 81152 MB)
[INFO] DynamicVRAM support detected and enabled
[INFO] Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
[INFO] ComfyUI version: 0.38.0
[INFO] comfy-aimdo version: 0.5.5
[INFO] comfy-kitchen version: 0.2.37
[INFO] comfyui-frontend-package version: 1.53.10
[INFO] comfyui-workflow-templates version: 0.11.76
[INFO] comfyui-embedded-docs version: 0.5.13
[INFO] comfy-kitchen version: 0.2.37
[INFO] comfy-aimdo version: 0.5.5
[INFO] [Prompt Server] web root: /usr/local/lib/python3.13/dist-packages/comfyui_frontend_package/static
[INFO] Asset seeder disabled
[INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


[INFO] NumExpr defaulting to 12 threads.
[ComfyUI-Easy-Use] server: v1.3.6 Loaded
[ComfyUI-Easy-Use] web root: /content/ComfyUI/custom_nodes/comfyui-easy-use/web_version/v2 Loaded
[INFO] ### Loading: ComfyUI-Manager (V3.42)
[INFO] [ComfyUI-Manager] network_mode: public
[INFO] [ComfyUI-Manager] ComfyUI per-queue preview override detected (PR #11261). Manager's preview method feature is disabled. Use ComfyUI's --preview-method CLI option or 'Settings > Execution > Live preview method'.
[INFO] ### ComfyUI Version: v0.38.0-32-ge9027f2b | Released on '2026-10-03'
[INFO] [ComfyUI-Manager] default cache updated: https://raw.githubusercontent.com/ltdrdata/ComfyUI-Manager/main/alter-list.json
[INFO] [ComfyUI-Manager] default cache updated: https://raw.githubusercontent.com/ltdrdata/ComfyUI-Manager/main/model-list.json
[INFO] [ComfyUI-Manager] default cache updated: https://raw.githubusercontent.com/ltdrdata/ComfyUI-Manager/main/github-stats.json
[INFO] [ComfyUI-Manager] default cache updated: h